# Diffusion Microscope — Experiment Runner

Runs all three experiments against GPT-2 and Pythia-410m and saves results to Google Drive.

**Runtime:** Select `Runtime → Change runtime type → T4 GPU` before running.

---
**Experiments:**
- **Exp 1 — Alpha compression visibility:** does alpha=1 produce more distinct images than alpha=1000? (both models, last layer)
- **Exp 2 — L0→L1 bottleneck:** what does the first transformer layer discard? (Pythia only, alpha=1, layers 0–3)
- **Exp 3 — Alpha sensitivity as novelty detector:** does LPIPS(alpha=1, alpha=1000) rank unusual prompts above common ones?

**Session persistence:** HuggingFace model cache and experiment results are stored on Drive — re-running skips completed work.

## 1 · Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU found — switch runtime to T4 GPU before continuing.')

In [ ]:
import os

# ── Edit these paths if needed ──────────────────────────────────────────────
REPO_URL    = 'https://github.com/leonorae/slicer'   # replace with your fork if needed
REPO_BRANCH = 'claude/analyze-experiment-confounders-uBu9h'
REPO_DIR    = '/content/slicer'
DRIVE_BASE  = '/content/drive/MyDrive/diffusion_microscope'
# ────────────────────────────────────────────────────────────────────────────

HF_CACHE_DIR    = os.path.join(DRIVE_BASE, 'hf_cache')
RESULTS_GPT2    = os.path.join(DRIVE_BASE, 'experiment_results_nb_gpt2')
RESULTS_PYTHIA  = os.path.join(DRIVE_BASE, 'experiment_results_nb_pythia')

os.makedirs(HF_CACHE_DIR,   exist_ok=True)
os.makedirs(RESULTS_GPT2,   exist_ok=True)
os.makedirs(RESULTS_PYTHIA, exist_ok=True)

os.environ['HF_HOME']               = HF_CACHE_DIR
os.environ['TRANSFORMERS_CACHE']    = HF_CACHE_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = HF_CACHE_DIR

print('Drive base :', DRIVE_BASE)
print('HF cache   :', HF_CACHE_DIR)
print('GPT-2 out  :', RESULTS_GPT2)
print('Pythia out :', RESULTS_PYTHIA)

## 2 · Clone repo and install

In [ ]:
import os

if os.path.isdir(REPO_DIR):
    print('Repo already cloned — pulling latest.')
    !git -C {REPO_DIR} fetch origin {REPO_BRANCH}
    !git -C {REPO_DIR} checkout {REPO_BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}
else:
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}

In [ ]:
# Colab already ships torch+CUDA — install everything else and avoid overwriting torch.
!pip install -q \
    open-clip-torch \
    diffusers \
    Pillow \
    lpips \
    datasets \
    nltk \
    sentencepiece \
    accelerate \
    scikit-learn \
    umap-learn

# Install the package itself (no deps — already installed above)
!pip install -q -e {REPO_DIR} --no-deps

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print('Install complete.')

## 3 · Patch configs with Drive output paths

The repo ships configs pointing to local `./experiment_results_nb_*` dirs.
This cell rewrites the output paths to point at Drive so results persist.

In [ ]:
import json, shutil

def patch_config(src_name, out_dir):
    src = os.path.join(REPO_DIR, src_name)
    with open(src) as f:
        cfg = json.load(f)
    cfg['output']['base_dir'] = out_dir
    dst = os.path.join(REPO_DIR, src_name)
    with open(dst, 'w') as f:
        json.dump(cfg, f, indent=2)
    print(f'{src_name} → {out_dir}')

patch_config('experiment_config_nb_gpt2.json',   RESULTS_GPT2)
patch_config('experiment_config_nb_pythia.json',  RESULTS_PYTHIA)

## 3.5 · Smoke test — validate new metrics before full run

Runs a minimal experiment (200-sample corpus, 1 probe, 4 seeds, alpha=1 and 10000)
and checks that the three new manifest fields are populated:
- `probe_text_stats` — n_tokens and perplexity per probe
- `seed_variance` — per-pixel variance across seeds per (proj, probe, layer, CFG)
- `probe_corpus_distances` — d_act and d_clip per (probe, projection, layer)

**Skip this cell if you're confident the code is correct and want to go straight to the full run.**

In [ ]:
%%time
import json, os, tempfile

SMOKE_DIR = os.path.join(DRIVE_BASE, '_smoke_test')
os.makedirs(SMOKE_DIR, exist_ok=True)

SMOKE_CFG = {
    "models": {
        "llm": "gpt2",
        "sd": "sd-legacy/stable-diffusion-v1-5",
        "clip_model": "ViT-L-14",
        "clip_pretrained": "openai"
    },
    "projections": {
        "types": ["per_layer"],
        "alpha_values": [1, 10000],
        "training_data_size": 200
    },
    "probe_texts": {"test": ["a cat"]},
    "layers": [11],
    "cfg_values": [7.5],
    "seeds": [42, 123, 777, 456],   # ≥4 needed for seed_variance
    "output": {"base_dir": SMOKE_DIR, "image_format": "png"},
    "use_auto_corpus": True
}

smoke_cfg_path = os.path.join(REPO_DIR, '_smoke_config.json')
with open(smoke_cfg_path, 'w') as f:
    json.dump(SMOKE_CFG, f, indent=2)

!cd {REPO_DIR} && python run_experiment.py --config _smoke_config.json --phase train
!cd {REPO_DIR} && python run_experiment.py --config _smoke_config.json --phase generate

# ── Validate manifest fields ─────────────────────────────────────────────────
manifest_path = os.path.join(SMOKE_DIR, 'manifest.json')
with open(manifest_path) as f:
    manifest = json.load(f)

errors = []

# Check probe_text_stats
pts = manifest.get('probe_text_stats', {})
if not pts:
    errors.append('FAIL: probe_text_stats is empty')
else:
    for slug, stats in pts.items():
        if stats.get('n_tokens') is None:
            errors.append(f'FAIL: probe_text_stats[{slug}].n_tokens is None')
        if 'perplexity' not in stats:
            errors.append(f'FAIL: probe_text_stats[{slug}].perplexity missing')
        else:
            print(f'  probe_text_stats[{slug}]: n_tokens={stats["n_tokens"]}, perplexity={stats.get("perplexity")}')

# Check seed_variance
sv = manifest.get('seed_variance', {})
if not sv:
    errors.append('FAIL: seed_variance is empty (need ≥4 seeds per group)')
else:
    for k, v in list(sv.items())[:3]:
        print(f'  seed_variance[{k}]: mean_pixel_var={v.get("mean_pixel_var"):.2f}, n_seeds={v.get("n_seeds")}')

# Check probe_corpus_distances
pcd = manifest.get('probe_corpus_distances', {})
if not pcd:
    errors.append('FAIL: probe_corpus_distances is empty')
else:
    for slug, proj_dict in list(pcd.items())[:1]:
        for proj_key, layer_dict in list(proj_dict.items())[:1]:
            for layer, dists in list(layer_dict.items())[:1]:
                print(f'  probe_corpus_distances[{slug}][{proj_key}][{layer}]: d_act={dists.get("d_act")}')

print()
if errors:
    for e in errors:
        print(e)
    print('\nSmoke test FAILED — check errors above before running full experiment.')
else:
    print('Smoke test PASSED — all new manifest fields populated correctly.')

## 4 · GPT-2 experiments

Covers **Exp 1** (alpha compression) and **Exp 3** (novelty detector).

Phases: `train` → `generate` → `grids` → `dashboard`

All phases are idempotent — if interrupted, re-run the cell and it will pick up from where it left off.

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_gpt2.json \
    --phase train

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_gpt2.json \
    --phase generate

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_gpt2.json \
    --phase grids

In [ ]:
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_gpt2.json \
    --phase dashboard
print(f'\nDashboard saved to Drive — download to view:\n  {RESULTS_GPT2}/dashboard.html')

## 5 · Pythia-410m experiments

Covers **Exp 1** (alpha compression, L23), **Exp 2** (L0→L1 bottleneck, L0-3), and **Exp 3** (novelty detector, L23).

Pythia-410m is ~800 MB. First run downloads it to the Drive HF cache.

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_pythia.json \
    --phase train

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_pythia.json \
    --phase generate

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_pythia.json \
    --phase grids

In [ ]:
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_pythia.json \
    --phase dashboard
print(f'\nDashboard saved to Drive — download to view:\n  {RESULTS_PYTHIA}/dashboard.html')

## 6 · Results

Run from here to review existing results without re-running the experiment.

In [ ]:
import pathlib, re, glob, json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# ── Load manifests for metric inspection ─────────────────────────────────────
def load_manifest(results_dir):
    p = pathlib.Path(results_dir) / 'manifest.json'
    if p.exists():
        with open(p) as f:
            return json.load(f)
    return {}

gpt2_manifest   = load_manifest(RESULTS_GPT2)
pythia_manifest = load_manifest(RESULTS_PYTHIA)

# ── Discover probe slugs from actual files on disk ───────────────────────────
def discover_slugs(results_dir):
    """Return sorted list of probe slugs that actually have images."""
    p = pathlib.Path(results_dir) / 'grids' / 'by_projection'
    if not p.exists():
        return []
    slugs = set()
    for proj_dir in p.iterdir():
        if proj_dir.is_dir():
            for slug_dir in proj_dir.iterdir():
                if slug_dir.is_dir():
                    slugs.add(slug_dir.name)
    return sorted(slugs)

gpt2_slugs   = discover_slugs(RESULTS_GPT2)
pythia_slugs = discover_slugs(RESULTS_PYTHIA)

print('GPT-2 slugs   :', gpt2_slugs or '(none found — run section 4 first)')
print('Pythia slugs  :', pythia_slugs or '(none found — run section 5 first)')

# ── Classify each slug into a tier ───────────────────────────────────────────
def classify_slug(slug):
    if any(w in slug for w in ['remembering', 'tuesday', 'midnight']):
        return 'unusual'
    if any(w in slug for w in ['cat', 'dog', 'house', 'tree']):
        return 'concrete'
    if any(w in slug for w in ['democracy', 'justice', 'entropy', 'grief', 'beauty', 'irony']):
        return 'abstract'
    return None

# ── Image lookup helpers ──────────────────────────────────────────────────────
def img_path(results_dir, alpha, slug, layer, cfg=7.5, seed=42):
    p = (pathlib.Path(results_dir) / 'grids' / 'by_projection'
         / f'per_layer_alpha{alpha}' / slug / 'per_layer'
         / f'L{layer:04d}_CFG{cfg:g}_seed{seed}.png')
    return p if p.exists() else None

def grid_path(results_dir, alpha, slug, seed=42):
    p = (pathlib.Path(results_dir) / 'grids' / 'by_projection'
         / f'per_layer_alpha{alpha}' / slug
         / f'grid_seed{seed}.png')
    return p if p.exists() else None

print('\nSlug tiers:')
for s in gpt2_slugs:
    print(f'  {classify_slug(s) or "unclassified":10s}  {s}')


### Confounder controls

Before interpreting any LPIPS results, inspect the per-probe control metrics stored in the manifest.

- **`n_tokens`** — prompt length. Unusual tier probes are systematically longer, which
  could drive LPIPS differences independent of semantics.
- **`perplexity`** — LLM's surprise at the probe text. High perplexity = far from pretraining
  distribution. Compare GPT-2 vs Pythia perplexity for the same probe: large differences
  mean the two models experience the probe very differently, confounding cross-model comparisons.
- **`d_act`** — normalised distance from corpus LLM-activation centroid. High = probe is
  out-of-distribution relative to the Ridge training corpus. If d_act is high and LPIPS is high,
  the LPIPS signal may reflect corpus-distance compression rather than semantic unusualness.
- **`seed_variance`** — mean per-pixel variance across seed images at fixed CLIP vector.
  If variance is similar at alpha=1 and alpha=1000 for the same probe, LPIPS differences
  genuinely reflect CLIP-vector differences. If variance is *higher* at alpha=1, the diffusion
  prior is doing more work there, and LPIPS may be inflated by noise rather than signal.

In [ ]:
def show_probe_controls(manifest, model_label, slugs, last_layer, alphas=(1, 1000, 10000)):
    """Print confounder metrics per probe as a formatted table."""
    pts  = manifest.get('probe_text_stats', {})
    pcd  = manifest.get('probe_corpus_distances', {})
    sv   = manifest.get('seed_variance', {})

    if not pts:
        print(f'  {model_label}: no probe_text_stats in manifest (run generate phase first)')
        return

    tier_order = ['concrete', 'abstract', 'unusual', None]
    header = f"{'slug':42s}  {'tier':8s}  {'tok':>4s}  {'ppl':>7s}  {'d_act':>6s}  " + \
             '  '.join(f'var(a={a})' for a in alphas)
    print(f'\n=== {model_label} — confounder controls ===')
    print(header)
    print('-' * len(header))

    # Sort by tier order
    def tier_key(s):
        t = classify_slug(s)
        return (tier_order.index(t) if t in tier_order else len(tier_order), s)

    for slug in sorted(slugs, key=tier_key):
        tier = classify_slug(slug) or '?'
        stats = pts.get(slug, {})
        n_tok = stats.get('n_tokens', '?')
        ppl   = stats.get('perplexity')
        ppl_s = f'{ppl:7.1f}' if isinstance(ppl, float) else '      ?'

        # d_act: take mean across projection keys (usually one)
        d_act_vals = []
        for proj_key, layers in pcd.get(slug, {}).items():
            v = layers.get(str(last_layer)) or layers.get(last_layer)
            if v and 'd_act' in v:
                d_act_vals.append(v['d_act'])
        d_act_s = f'{np.mean(d_act_vals):6.2f}' if d_act_vals else '     ?'

        # seed_variance per alpha at this layer
        var_parts = []
        for alpha in alphas:
            proj_key = f'per_layer_alpha{alpha}'
            vk = f'{proj_key}/{slug}/L{last_layer:04d}/CFG7.5'
            entry = sv.get(vk, {})
            mpv = entry.get('mean_pixel_var')
            var_parts.append(f'{mpv:9.1f}' if isinstance(mpv, float) else '        ?')

        print(f'{slug:42s}  {tier:8s}  {str(n_tok):>4s}  {ppl_s}  {d_act_s}  {"  ".join(var_parts)}')

show_probe_controls(gpt2_manifest,   'GPT-2',      gpt2_slugs,   last_layer=11)
show_probe_controls(pythia_manifest, 'Pythia-410m', pythia_slugs, last_layer=23)


### Exp 1 — Alpha compression: does alpha=1 produce more distinct images?

**What's being tested:** High alpha (1000) regularises the Ridge projection heavily,
pulling all probe vectors toward the corpus mean — like lossy compression in embedding space.
Low alpha (1) preserves relative distances but amplifies noise in low-variance directions.

**What to look for:** Each figure shows the *same prompt* at alpha=1 (top row) vs alpha=1000 (bottom row).
- **Concrete prompts** (cat, dog, house, tree) should look *similar* across alphas — their
  signal is near the corpus mean, so compression doesn't destroy much.
- **Abstract prompts** (democracy, grief, entropy) should look *more different* — their
  distinguishing information lives in low-variance directions that high alpha kills.

The nn_recall@5 metric predicted this: last-layer nn_recall at alpha=1000 is lower for
abstract prompts (~0.35) than concrete (~0.5), meaning the compressor loses more
neighbourhood structure for abstract concepts.

### Primary metric: cosine distance in CLIP space

LPIPS depends on CFG scale, the diffusion prior, and AlexNet's perceptual model — none of
which are controlled by the experiment. The actual question ("does alpha compression destroy
information in the CLIP projection?") is answerable directly in CLIP space.

**`cosine_distance(proj_α1(act), proj_α1000(act))`** for the same probe is the primary metric:
- CFG-independent (no SD involved)
- Directly geometric: exactly "how far did the CLIP vector move under compression"
- Comparable across models and layers on a common scale [0, 2]

LPIPS is then a secondary check: given a measured CLIP distance, does it exceed the
CFG=7.5 detection threshold and show up visually?  Plotting LPIPS vs. cosine distance
characterises that threshold.  If probes with large CLIP distances produce low LPIPS,
the images are not a reliable diagnostic at this CFG setting.

The seed variance floor is the lower bound on LPIPS — below it, you cannot distinguish
conditioning signal from diffusion prior noise.

In [ ]:
from scipy.spatial.distance import cosine as cosine_dist

def clip_cosine_distances(manifest, slugs, last_layer, alpha_pairs=ALPHA_PAIRS):
    """
    For each probe and each (alpha_a, alpha_b) pair, compute cosine distance
    between the two projected CLIP vectors stored in manifest["probe_clip_vectors"].

    Returns: {(alpha_a, alpha_b): {slug: cosine_distance}}
    """
    vecs = manifest.get('probe_clip_vectors', {})
    results = {pair: {} for pair in alpha_pairs}

    for alpha_a, alpha_b in alpha_pairs:
        key_a = f'per_layer_alpha{alpha_a}'
        key_b = f'per_layer_alpha{alpha_b}'
        layer_s = str(last_layer)

        for slug in slugs:
            va = vecs.get(key_a, {}).get(slug, {}).get(layer_s)
            vb = vecs.get(key_b, {}).get(slug, {}).get(layer_s)
            if va is not None and vb is not None:
                results[(alpha_a, alpha_b)][slug] = cosine_dist(va, vb)

    return results


gpt2_clip_dists   = clip_cosine_distances(gpt2_manifest,   gpt2_slugs,   last_layer=11)
pythia_clip_dists = clip_cosine_distances(pythia_manifest, pythia_slugs, last_layer=23)

# ── Primary metric: bar chart by tier ────────────────────────────────────────
n_models = 2
n_pairs  = len(ALPHA_PAIRS)
fig, axes = plt.subplots(n_models, n_pairs, figsize=(5 * n_pairs, 4.5 * n_models),
                         squeeze=False)

for row, (model_label, clip_dists, slugs) in enumerate([
    ('GPT-2 L11',       gpt2_clip_dists,   gpt2_slugs),
    ('Pythia-410m L23', pythia_clip_dists, pythia_slugs),
]):
    for col, (alpha_a, alpha_b) in enumerate(ALPHA_PAIRS):
        ax = axes[row][col]
        scores = clip_dists.get((alpha_a, alpha_b), {})

        tier_vals = {t: [] for t in tier_order}
        for slug, v in scores.items():
            t = classify_slug(slug)
            if t in tier_vals:
                tier_vals[t].append(v)

        means = [np.mean(tier_vals[t]) if tier_vals[t] else 0 for t in tier_order]
        stds  = [np.std(tier_vals[t])  if len(tier_vals[t]) > 1 else 0 for t in tier_order]

        ax.bar(tier_order, means, yerr=stds, color=tier_colors,
               alpha=0.85, capsize=5, edgecolor='white', linewidth=0.5)
        for t_idx, tier in enumerate(tier_order):
            for v in tier_vals[tier]:
                ax.plot(t_idx, v, 'o', color='white', markersize=5, alpha=0.8)

        ax.set_title(f'{model_label}\ncosine dist(α={alpha_a}, α={alpha_b})', fontsize=9)
        ax.set_ylabel('cosine distance in CLIP space' if col == 0 else '')
        ax.set_ylim(0, max(max(means) * 1.5, 0.01) if any(means) else 0.1)
        ax.grid(True, alpha=0.25, axis='y')

plt.suptitle('PRIMARY METRIC — CLIP cosine distance between projections at different alphas\n'
             'CFG-independent. Tier ordering here = genuine compression effect.',
             fontsize=10)
plt.tight_layout()
plt.show()

# Print raw values grouped by tier
for model_label, clip_dists, slugs in [
    ('GPT-2 L11',       gpt2_clip_dists,   gpt2_slugs),
    ('Pythia-410m L23', pythia_clip_dists, pythia_slugs),
]:
    print(f'\n{model_label}')
    for (aa, ab), scores in clip_dists.items():
        print(f'  α={aa} vs α={ab}:')
        for slug in sorted(scores, key=lambda s: classify_slug(s) or 'z'):
            print(f'    {classify_slug(slug) or "?":10s}  {slug:42s}  {scores[slug]:.4f}')


In [ ]:
ALPHAS = [1, 1000, 10000]  # all alpha values in the experiment

def show_alpha_comparison(results_dir, layer, slugs, title='', seeds=(42, 123, 777),
                          alphas=ALPHAS):
    """One row per alpha, one column per slug.
    Uses the first seed that has images for all alphas."""
    available = []
    chosen_paths = {}  # slug → {alpha: path}
    for slug in slugs:
        for seed in seeds:
            paths = {a: img_path(results_dir, a, slug, layer, seed=seed) for a in alphas}
            if all(p is not None for p in paths.values()):
                available.append(slug)
                chosen_paths[slug] = paths
                break

    if not available:
        print(f'No images found in {results_dir} for layer {layer} — check paths.')
        return

    n_rows, n_cols = len(alphas), len(available)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3.5 * n_rows + 0.5),
                             squeeze=False)
    fig.suptitle(title, fontsize=11, y=1.01)

    for col, slug in enumerate(available):
        for row, alpha in enumerate(alphas):
            ax = axes[row][col]
            p = chosen_paths[slug][alpha]
            ax.imshow(Image.open(p))
            ax.axis('off')
            if row == 0:
                ax.set_title(slug.replace('_', ' '), fontsize=8)
            if col == 0:
                ax.set_ylabel(f'α = {alpha}', fontsize=9)

    plt.tight_layout()
    plt.show()

# Split GPT-2 slugs into concrete vs abstract
gpt2_concrete = [s for s in gpt2_slugs if classify_slug(s) == 'concrete']
gpt2_abstract = [s for s in gpt2_slugs if classify_slug(s) == 'abstract']

print('=== GPT-2 — Layer 11 (last layer) ===')
show_alpha_comparison(RESULTS_GPT2, layer=11, slugs=gpt2_concrete,
                      title='GPT-2 L11 — concrete prompts — α=1, 1000, 10000')
show_alpha_comparison(RESULTS_GPT2, layer=11, slugs=gpt2_abstract,
                      title='GPT-2 L11 — abstract prompts — α=1, 1000, 10000')


In [ ]:
pythia_concrete = [s for s in pythia_slugs if classify_slug(s) == 'concrete']
pythia_abstract = [s for s in pythia_slugs if classify_slug(s) == 'abstract']

print('=== Pythia-410m — Layer 23 (last layer) ===')
show_alpha_comparison(RESULTS_PYTHIA, layer=23, slugs=pythia_concrete,
                      title='Pythia L23 — concrete prompts — alpha=1 vs 1000')
show_alpha_comparison(RESULTS_PYTHIA, layer=23, slugs=pythia_abstract,
                      title='Pythia L23 — abstract prompts — alpha=1 vs 1000')

### Exp 2 — L0→L1 bottleneck in Pythia

**What's being tested:** At alpha=1000, Pythia L0 has the *highest* erank (494) and L1 dips
to 478 before subsequent layers rise again. This is the opposite of GPT-2 (where L0 is
singular/rank-deficient). The hypothesis: Pythia's embedding layer is maximally
diffuse, and the first transformer layer *compresses* it before later layers re-expand.

**What to look for:** Does L0 look more diffuse or generic than L1?
If the bottleneck hypothesis is right, L0 images should look less structured than L1 —
more like random textures or colour fields — because L0 has no transformer computation
behind it, just raw token embeddings.

**Important caveat:** the erank dip only appears at alpha=1000. At alpha=1 (shown
below), L0 has the *lowest* erank (354) and rises monotonically. The experiment tests
both alphas to distinguish architectural signal from regularisation artefact.

In [ ]:
def show_layer_sweep(results_dir, alpha, layers, slug, title='', seeds=(42, 123, 777)):
    """One image per layer for a single probe."""
    valid = []
    for L in layers:
        for seed in seeds:
            p = img_path(results_dir, alpha, slug, L, seed=seed)
            if p:
                valid.append((L, p))
                break

    if not valid:
        print(f'No images for slug "{slug}" at alpha={alpha} in {results_dir}')
        return

    fig, axes = plt.subplots(1, len(valid), figsize=(3 * len(valid), 4), squeeze=False)
    fig.suptitle(title or f'"{slug.replace("_", " ")}" — L{layers[0]}→L{layers[-1]} at alpha={alpha}',
                 fontsize=10)
    for i, (L, p) in enumerate(valid):
        axes[0][i].imshow(Image.open(p))
        axes[0][i].axis('off')
        axes[0][i].set_title(f'L{L}', fontsize=9)
    plt.tight_layout()
    plt.show()

# Pick one concrete and one abstract probe for the sweep
ref_concrete = next((s for s in pythia_slugs if classify_slug(s) == 'concrete'), None)
ref_abstract = next((s for s in pythia_slugs if classify_slug(s) == 'abstract'), None)

for slug in [ref_concrete, ref_abstract]:
    if not slug:
        continue
    print(f'\n--- "{slug.replace("_", " ")}" ---')
    show_layer_sweep(RESULTS_PYTHIA, alpha=1,    layers=[0, 1, 2, 3],
                     title=f'"{slug.replace("_", " ")}" at alpha=1 — raw embedding → first 3 transformer layers')
    show_layer_sweep(RESULTS_PYTHIA, alpha=1000, layers=[0, 1, 2, 3],
                     title=f'"{slug.replace("_", " ")}" at alpha=1000 — same layers with heavy compression')

### Exp 3 — Alpha sensitivity as novelty detector

**What's being tested:** Do unusual prompts change *more* between alpha=1 and alpha=1000
than common ones? The hypothesis: unusual prompts have their distinguishing information
concentrated in low-variance directions that high alpha kills. Common prompts live near
the corpus mean — their projection is insensitive to the compression.

**Tiers:**
- *Common-concrete* (should be alpha-insensitive): cat, dog, house, tree
- *Common-abstract* (moderate sensitivity): democracy, justice, beauty, grief
- *Unusual* (should be alpha-sensitive): "the feeling of almost remembering",
  "the color of Tuesday", "entropy at midnight"

**What to look for:** LPIPS(alpha=1 image, alpha=1000 image) — higher = more different.
If the hypothesis holds, the bar chart should show: unusual > abstract > concrete.
If the bars are flat across tiers, compression is not tier-sensitive and the
alpha-as-novelty-detector idea doesn't hold for this corpus and model.

**Confound to watch:** low-variance directions in the projection ≠ semantically unusual
prompts. They could just be directions underrepresented in the training corpus.

In [ ]:
import torch
import lpips as lpips_lib

_lpips_fn = None

def get_lpips():
    global _lpips_fn
    if _lpips_fn is None:
        _lpips_fn = lpips_lib.LPIPS(net='alex', verbose=False)
    return _lpips_fn

def compute_lpips(path_a, path_b):
    """LPIPS perceptual distance between two image files."""
    def load_tensor(p):
        img = Image.open(p).convert('RGB').resize((256, 256))
        t = torch.tensor(np.array(img), dtype=torch.float32)
        t = t.permute(2, 0, 1) / 127.5 - 1.0  # [-1, 1]
        return t.unsqueeze(0)
    fn = get_lpips()
    with torch.no_grad():
        return fn(load_tensor(path_a), load_tensor(path_b)).item()

def lpips_alpha_pair(results_dir, layer, slug, alpha_a, alpha_b,
                     seeds=(42, 123, 777, 456, 789, 101, 234, 567,
                            890, 111, 222, 333, 444, 555, 666, 999)):
    """Mean LPIPS between alpha_a and alpha_b images across available seeds."""
    vals = []
    for seed in seeds:
        pa = img_path(results_dir, alpha_a, slug, layer, seed=seed)
        pb = img_path(results_dir, alpha_b, slug, layer, seed=seed)
        if pa and pb:
            vals.append(compute_lpips(str(pa), str(pb)))
    return np.mean(vals) if vals else None

# Alpha pairs to compare
ALPHA_PAIRS = [(1, 1000), (1, 10000), (1000, 10000)]

tier_order  = ['concrete', 'abstract', 'unusual']
tier_colors = ['#4CAF50', '#FF9800', '#F44336']

# results_lpips_pairs[(model_label, alpha_a, alpha_b)][slug] = lpips_value
results_lpips_pairs = {}

for model_label, results_dir, last_layer, slugs in [
    ('GPT-2 L11',       RESULTS_GPT2,   11, gpt2_slugs),
    ('Pythia-410m L23', RESULTS_PYTHIA, 23, pythia_slugs),
]:
    for alpha_a, alpha_b in ALPHA_PAIRS:
        key = (model_label, alpha_a, alpha_b)
        print(f'Computing LPIPS({alpha_a}, {alpha_b}) for {model_label}...')
        scores = {}
        for slug in slugs:
            v = lpips_alpha_pair(results_dir, last_layer, slug, alpha_a, alpha_b)
            if v is not None:
                scores[slug] = v
                tier = classify_slug(slug) or 'other'
                print(f'  {tier:10s}  {slug:42s}  LPIPS = {v:.4f}')
        results_lpips_pairs[key] = scores

# Keep backward-compatible alias for existing cells
results_lpips = {k[0]: v for k, v in results_lpips_pairs.items() if k[1] == 1 and k[2] == 1000}
print('\nDone.')


In [ ]:
models_list = ['GPT-2 L11', 'Pythia-410m L23']
n_models = len(models_list)
n_pairs  = len(ALPHA_PAIRS)

fig, axes = plt.subplots(n_models, n_pairs,
                         figsize=(5 * n_pairs, 4.5 * n_models),
                         squeeze=False)

for row, model_label in enumerate(models_list):
    for col, (alpha_a, alpha_b) in enumerate(ALPHA_PAIRS):
        ax = axes[row][col]
        scores = results_lpips_pairs.get((model_label, alpha_a, alpha_b), {})

        tier_vals = {t: [] for t in tier_order}
        for slug, v in scores.items():
            t = classify_slug(slug)
            if t in tier_vals:
                tier_vals[t].append(v)

        means = [np.mean(tier_vals[t]) if tier_vals[t] else 0 for t in tier_order]
        stds  = [np.std(tier_vals[t])  if len(tier_vals[t]) > 1 else 0 for t in tier_order]

        ax.bar(tier_order, means, yerr=stds, color=tier_colors,
               alpha=0.85, capsize=5, edgecolor='white', linewidth=0.5)

        for t_idx, tier in enumerate(tier_order):
            for v in tier_vals[tier]:
                ax.plot(t_idx, v, 'o', color='white', markersize=4, alpha=0.7)

        ax.set_title(f'{model_label}\nLPIPS(α={alpha_a}, α={alpha_b})', fontsize=9)
        ax.set_ylabel('LPIPS' if col == 0 else '')
        ax.set_ylim(0, max(max(means) * 1.4, 0.05) if any(means) else 0.1)
        ax.grid(True, alpha=0.25, axis='y')

plt.suptitle('Alpha compression LPIPS by tier — all pairs\n'
             'Hypothesis: unusual > abstract > concrete for (1 vs 1000); '
             'everything converges at (1000 vs 10000)',
             fontsize=10)
plt.tight_layout()
plt.show()


#### CFG sensitivity floor: LPIPS vs. CLIP cosine distance

If LPIPS is a reliable proxy for CLIP vector differences at CFG=7.5, these should
correlate.  If the scatter is flat or noisy, CFG is not amplifying the CLIP differences
into the visible regime — the images are not a reliable diagnostic.

The seed variance floor (dashed line per model) marks the minimum LPIPS below which
the diffusion prior is filling in the image regardless of conditioning.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), squeeze=False)
tier_markers = {'concrete': 'o', 'abstract': 's', 'unusual': '^'}

for col, (model_label, clip_dists, lpips_scores, manifest, last_layer) in enumerate([
    ('GPT-2 L11',       gpt2_clip_dists,   results_lpips.get('GPT-2 L11', {}),
     gpt2_manifest,   11),
    ('Pythia-410m L23', pythia_clip_dists, results_lpips.get('Pythia-410m L23', {}),
     pythia_manifest, 23),
]):
    ax = axes[0][col]
    cd_scores = clip_dists.get((1, 1000), {})

    xs, ys, colors, markers, labels = [], [], [], [], []
    for slug in cd_scores:
        cd  = cd_scores.get(slug)
        lp  = lpips_scores.get(slug)
        tier = classify_slug(slug) or 'other'
        if cd is not None and lp is not None:
            xs.append(cd)
            ys.append(lp)
            colors.append({'concrete': '#4CAF50', 'abstract': '#FF9800',
                           'unusual': '#F44336'}.get(tier, '#999'))
            markers.append(tier_markers.get(tier, 'o'))
            labels.append(f'{slug[:20]}\n({tier})')

    for x, y, c, m, lab in zip(xs, ys, colors, markers, labels):
        ax.scatter(x, y, color=c, marker=m, s=80, alpha=0.85, zorder=3)
        ax.annotate(lab, (x, y), fontsize=6, xytext=(4, 2), textcoords='offset points')

    # Seed variance floor: mean seed variance across all probes at alpha=1
    sv = manifest.get('seed_variance', {})
    floor_vals = []
    for k, v in sv.items():
        # Only alpha=1 entries at this layer, normalised to [0,1] pixel range
        if f'per_layer_alpha1' in k and f'L{last_layer:04d}' in k:
            mpv = v.get('mean_pixel_var', 0)
            floor_vals.append(mpv / (255**2))  # pixel var → approximate LPIPS scale
    if floor_vals:
        ax.axhline(np.mean(floor_vals), linestyle='--', color='white',
                   alpha=0.5, linewidth=1, label=f'~seed var floor (α=1)')

    # Correlation
    if len(xs) > 2:
        r = np.corrcoef(xs, ys)[0, 1]
        ax.set_title(f'{model_label}\nLPIPS vs CLIP cosine dist (α=1→1000),  r={r:.2f}',
                     fontsize=9)
    else:
        ax.set_title(f'{model_label}\nLPIPS vs CLIP cosine dist (α=1→1000)', fontsize=9)

    ax.set_xlabel('cosine distance in CLIP space (α=1 vs α=1000)')
    ax.set_ylabel('LPIPS (α=1 vs α=1000 images)')
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=7)

# Legend for tiers
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#4CAF50', markersize=8, label='concrete'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#FF9800', markersize=8, label='abstract'),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='#F44336', markersize=8, label='unusual'),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, 1.0))
plt.suptitle('CFG sensitivity floor: does LPIPS track CLIP distance?\n'
             'Flat scatter = images are not a reliable diagnostic at CFG=7.5',
             fontsize=10, y=1.06)
plt.tight_layout()
plt.show()


In [ ]:
# Side-by-side images for highest and lowest LPIPS probes
for model_label, results_dir, last_layer, slugs in [
    ('GPT-2 L11',       RESULTS_GPT2,   11, gpt2_slugs),
    ('Pythia-410m L23', RESULTS_PYTHIA, 23, pythia_slugs),
]:
    scores = results_lpips.get(model_label, {})
    if not scores:
        continue
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    print(f'\n=== {model_label} — highest and lowest LPIPS pairs ===')
    for slug, v in ranked[:2] + ranked[-2:]:
        show_alpha_comparison(results_dir, last_layer, [slug],
                              title=f'"{slug.replace("_", " ")}" — LPIPS={v:.3f} ({classify_slug(slug)})')

### Full grid PNGs — best overview

Each grid image shows **all layers × all seeds** for a single probe at one alpha setting.
This is the most information-dense view — good for spotting whether layers produce
coherent variation or whether the images look random.

In [ ]:
from IPython.display import display as ipy_display

def show_grid_images(results_dir, alpha, model_label):
    pattern = str(pathlib.Path(results_dir) / 'grids' / 'by_projection'
                  / f'per_layer_alpha{alpha}' / '*' / 'grid_seed*.png')
    grid_paths = sorted(glob.glob(pattern))
    if not grid_paths:
        print(f'No grid PNGs found for {model_label} alpha={alpha}')
        return
    for p in grid_paths:
        slug = pathlib.Path(p).parent.name
        tier = classify_slug(slug) or 'other'
        print(f'  [{tier}]  {slug.replace("_", " ")}  (alpha={alpha})')
        ipy_display(Image.open(p))

for alpha in [1, 1000]:
    print(f'\n══ GPT-2   alpha={alpha} ══════════════════════════════════════')
    show_grid_images(RESULTS_GPT2, alpha, 'GPT-2')

for alpha in [1, 1000]:
    print(f'\n══ Pythia  alpha={alpha} ══════════════════════════════════════')
    show_grid_images(RESULTS_PYTHIA, alpha, 'Pythia-410m')

## 7 · Save summary

Results are already on Drive (output dirs point there). This cell writes a short summary of what ran.

In [ ]:
import datetime

summary_path = os.path.join(DRIVE_BASE, 'run_summary.txt')
lines = [
    f'Run completed: {datetime.datetime.now().isoformat()}',
    f'Branch: {REPO_BRANCH}',
    '',
    'GPT-2 results:',
]
for root, dirs, files in os.walk(RESULTS_GPT2):
    png_count = sum(1 for f in files if f.endswith('.png'))
    if png_count:
        lines.append(f'  {os.path.relpath(root, RESULTS_GPT2)}: {png_count} images')

lines.append('')
lines.append('Pythia-410m results:')
for root, dirs, files in os.walk(RESULTS_PYTHIA):
    png_count = sum(1 for f in files if f.endswith('.png'))
    if png_count:
        lines.append(f'  {os.path.relpath(root, RESULTS_PYTHIA)}: {png_count} images')

if results_lpips:
    lines.append('')
    lines.append('Exp 3 LPIPS summary (alpha=1 vs alpha=1000):')
    for model_label, scores in results_lpips.items():
        lines.append(f'  {model_label}:')
        for slug, v in sorted(scores.items(), key=lambda x: -x[1]):
            tier = classify_slug(slug) or 'other'
            lines.append(f'    {tier:10s} {slug:42s} {v:.4f}')

summary = '\n'.join(lines)
print(summary)
with open(summary_path, 'w') as f:
    f.write(summary)
print(f'\nSummary written to {summary_path}')